# Credit Card Fraud Detection

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os

path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
df = pd.read_csv(os.path.join(path, 'creditcard.csv'))
print(df.shape)

## EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print(df.head())
print(df['Class'].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=df)
plt.title('Class Distribution')
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(df['Amount'], bins=50, kde=True)
plt.title('Transaction Amount Distribution')
plt.show()

## Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = df.dropna()
X = df.drop(columns=['Class'])
y = df['Class']

scaler = StandardScaler()
X[['Amount', 'Time']] = scaler.fit_transform(X[['Amount', 'Time']])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## SMOTE Balancing

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print('Resampled:', X_train_res.shape, y_train_res.value_counts().to_dict())

## Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd

iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_train_res)
iso_train = iso.decision_function(X_train_res)
iso_test = iso.decision_function(X_test)

X_train_feat = pd.DataFrame(X_train_res).copy()
X_test_feat = pd.DataFrame(X_test).copy()
X_train_feat['iso_score'] = iso_train
X_test_feat['iso_score'] = iso_test

## Local Outlier Factor

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01, novelty=True, n_jobs=-1)
lof.fit(X_train_res)
lof_train = lof.decision_function(X_train_res)
lof_test = lof.decision_function(X_test)

X_train_feat['lof_score'] = lof_train
X_test_feat['lof_score'] = lof_test
print('Feature shape:', X_train_feat.shape)

## Train XGBoost

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_feat, y_train_res)

## Evaluation

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test_feat)
y_prob = model.predict_proba(X_test_feat)[:, 1]

print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

## ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}', color='darkorange', lw=2)
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

## Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Fraud', 'Fraud'])
fig, ax = plt.subplots(figsize=(6,5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Confusion Matrix')
plt.show()